1. Importing Libraries

In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.stats import spearmanr, chi2_contingency,gaussian_kde
import re
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

2. Basic Inception

In [30]:
df=pd.read_csv('pnm_experience_dump_2026-03-09-1530.csv')
print("shape:", df.shape)
print("\ncolumn dtypes:\n", df.dtypes)

shape: (4744, 57)

column dtypes:
 ORDER_ID                           int64
VENDOR_ID                          int64
SUPERVISOR_ID                      int64
PICKUP_CITY_NAME                     str
PICKUP_GEO_REGION_ID               int64
PICKUP_ZONE_ID                   float64
PICKUP_ZONE_NAME                     str
DROP_CITY_NAME                       str
DROP_GEO_REGION_ID                 int64
DROP_ZONE_ID                     float64
DROP_ZONE_NAME                       str
ORDER_STATUS                         str
ITEMS_LIST                           str
PACKAGE_NAME                         str
ADD_ONS                              str
ORDER_CREATED_TS_IST                 str
ORDER_UPDATED_TS_IST                 str
SHIFTING_TS_IST                      str
VENDOR_OWNER_ACCEPTED_TS_IST         str
SUPERVISOR_ACCEPTED_TS_IST           str
TRIP_STARTED_TS_IST                  str
SHIFTING_STARTED_TS_IST              str
PICKUP_COMPLETED_TS_IST              str
ORDER_COMPLETED_TS_IST

In [31]:
print("\nClassification split:")
print(df['CLASSIFICATION'].value_counts())
print(df['CLASSIFICATION'].value_counts(normalize=True).mul(100).round(1))



Classification split:
CLASSIFICATION
Promoter     3884
Detractor     513
Neutral       347
Name: count, dtype: int64
CLASSIFICATION
Promoter     81.9
Detractor    10.8
Neutral       7.3
Name: proportion, dtype: float64


In [32]:
missing = df.isnull().sum()
print("\nMissing values (only cols with >0):")
print(missing[missing > 0].sort_values(ascending=False))


Missing values (only cols with >0):
ORDER_CANCELLED_TS_IST           4744
ON_TIME_DELIVERY_FLAG            4744
OTA_BREACH_TAT_MINUTES           4429
AVG_RESOLUTION_TAT               2977
ISSUE_SUBISSUE_DICTIONARY        2976
ESCALATED_TO_CITY_TEAM_FLAG      2976
FIRST_CONTACT_RESOLUTION_FLAG    2976
ADD_ONS                           682
OTA_FLAG                          506
DROP_ZONE_NAME                    478
DROP_ZONE_ID                      478
SURGE_MULTIPLIER                  371
DRY_RUN_DISTANCE_KMS              191
PICKUP_ZONE_NAME                   67
PICKUP_ZONE_ID                     67
PICKUP_COMPLETED_TS_IST            13
SHIFTING_STARTED_TS_IST             4
ORDER_COMPLETED_TS_IST              3
TRIP_STARTED_TS_IST                 2
dtype: int64


In [33]:
df.describe().style.format("{:.2f}")

,ORDER_ID,VENDOR_ID,SUPERVISOR_ID,PICKUP_GEO_REGION_ID,PICKUP_ZONE_ID,DROP_GEO_REGION_ID,DROP_ZONE_ID,ORDER_CANCELLED_TS_IST,OTA_BREACH_TAT_MINUTES,PICKUP_FLOOR,DROP_FLOOR,FINAL_FARE,SURGE_MULTIPLIER,TOTAL_ORDER_FARE,TOTAL_TICKETS,VENDOR_SUPERVISOR_TICKETS,VENDOR_OWNER_TICKETS,CUSTOMER_TICKETS,DETRACTOR_TICKETS,PORTER_SUPPORT_TICKETS,SPRINKLR_TICKETS,FCR_TICKETS,SPRINKLR_SESSION_COUNT,AVG_RESOLUTION_TAT,DRY_RUN_DISTANCE_KMS,INITIAL_CFT,FINAL_CFT,ON_TIME_DELIVERY_FLAG
count,4744.00,4744.00,4744.00,4744.00,4677.00,4744.00,4266.00,0.00,315.00,4744.00,4744.00,4744.00,4373.00,4744.00,4744.00,4744.00,4744.00,4744.00,4744.00,4744.00,4744.00,4744.00,4744.00,1767.00,4553.00,4744.00,4744.00,0.00
mean,2284984.03,9626.06,24739.47,3.72,142.61,3.72,146.71,nan,51.66,1.03,1.01,5290.74,1.10,5300.14,0.79,0.11,0.10,0.46,0.12,0.00,0.00,0.35,1.43,1239.05,8.61,309.15,326.73,nan
std,38907.26,6129.14,10533.28,2.62,98.33,2.62,98.09,nan,124.11,1.98,1.93,3843.40,0.16,3850.49,1.46,0.39,0.38,0.99,0.33,0.03,0.00,0.73,3.67,3680.16,43.81,234.37,255.82,nan
min,2030307.00,71.00,112.00,1.00,1.00,1.00,1.00,nan,0.00,0.00,0.00,1019.15,0.80,1019.15,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-0.02,0.00,4.00,4.00,nan
25%,2254040.50,3199.25,17344.00,2.00,35.00,2.00,36.00,nan,7.00,0.00,0.00,2510.25,1.00,2513.12,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.08,132.00,132.00,nan
50%,2285229.50,10215.00,27775.00,3.00,174.00,3.00,178.00,nan,23.00,0.00,0.00,4233.50,1.10,4238.10,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,38.08,4.03,240.00,250.50,nan
75%,2318655.00,14646.00,34107.00,5.00,185.00,5.00,185.00,nan,54.50,2.00,2.00,6904.36,1.20,6915.90,1.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,420.56,11.81,434.25,461.00,nan
max,2355778.00,18229.00,36514.00,14.00,1631.00,14.00,1631.00,nan,1543.00,35.00,27.00,35839.50,2.00,35839.50,15.00,7.00,5.00,11.00,1.00,1.00,0.00,7.00,131.00,39197.13,1740.10,1555.00,1896.00,nan


3. Feature engineering

In [34]:
df['IS_DETRACTOR'] = (df['CLASSIFICATION'] == 'Detractor').astype(int)
class_map = {'Promoter': 0, 'Neutral': 1, 'Detractor': 2}
df['CLASS_NUM'] = df['CLASSIFICATION'].map(class_map)


In [35]:
df['FCR_FLAG']        = (df['FIRST_CONTACT_RESOLUTION_FLAG'] == 'Yes').astype(int)
df['OTA_BREACH']      = df['OTA_BREACH_TAT_MINUTES'].notna().astype(int)
df['IS_MODIFICATION'] = (df['IS_MODIFICATION_DONE'] == 'Yes').astype(int)
df['IS_PEAK']         = (df['PEAK_OR_NON_PEAK_DAYS'] == 'Peak Days').astype(int)
df['IS_NOLIFT']       = (df['PICKUP_LIFT_STATUS'] == 'NoLift').astype(int)
df['IS_SAME_DAY']     = (df['SAME_DAY_OR_SCHEDULED'] == 'Same Day').astype(int)

# -- Numeric conversions
df['PICKUP_FLOOR']    = pd.to_numeric(df['PICKUP_FLOOR'], errors='coerce').fillna(0)
df['DROP_FLOOR']      = pd.to_numeric(df['DROP_FLOOR'],   errors='coerce').fillna(0)
df['TOTAL_FLOOR']     = df['PICKUP_FLOOR'] + df['DROP_FLOOR']
df['DRY_RUN']         = df['DRY_RUN_DISTANCE_KMS'].fillna(0)
df['TAT_HRS']         = df['AVG_RESOLUTION_TAT'].fillna(0) / 60
df['CFT_DELTA']       = df['FINAL_CFT'] - df['INITIAL_CFT']
df['SURGE_MULTIPLIER']= df['SURGE_MULTIPLIER'].fillna(1)
vendor_map = {'New': 0, 'Bronze': 1, 'Silver': 2, 'Gold': 3, 'GoldPlus': 4}
df['VENDOR_TIER_NUM'] = df['VENDOR_BUCKET_TYPE'].map(vendor_map).fillna(0)
df['BHK_SIZE']        = df['PACKAGE_NAME'].str.extract(r'(\d)\s*BHK').astype(float).fillna(0)

# -- Issue subissue parsing
def extract_subissues(row):
    if pd.isna(row) or str(row) == '{}':
        return []
    parts = re.findall(r'[A-Za-z][A-Za-z /]+?:\s*([^;{}]+)', str(row))
    return [p.strip() for p in parts if len(p.strip()) > 2]

df['SUBISSUES']   = df['ISSUE_SUBISSUE_DICTIONARY'].apply(extract_subissues)
df['NUM_ISSUES']  = df['SUBISSUES'].apply(len)
df['HAS_DAMAGE']  = df['SUBISSUES'].apply(lambda x: 1 if 'Goods damaged' in x else 0)
df['HAS_CREW']    = df['SUBISSUES'].apply(lambda x: 1 if 'Crew Behaviour / Hygiene' in x else 0)
df['HAS_ADDON']   = df['SUBISSUES'].apply(lambda x: 1 if any('Add on' in i for i in x) else 0)
df['HAS_PAYMENT'] = df['SUBISSUES'].apply(lambda x: 1 if any('Extra amount' in i for i in x) else 0)
df['HAS_DELAY']   = df['SUBISSUES'].apply(lambda x: 1 if any('Delay' in i for i in x) else 0)
df['HAS_RNR']     = df['SUBISSUES'].apply(lambda x: 1 if any('RNR' in i or 'rnr' in i.lower() for i in x) else 0)

print("New columns added:")
new_cols = ['IS_DETRACTOR','CLASS_NUM','FCR_FLAG','OTA_BREACH','IS_MODIFICATION',
            'IS_PEAK','IS_NOLIFT','TOTAL_FLOOR','TAT_HRS','CFT_DELTA',
            'VENDOR_TIER_NUM','BHK_SIZE','NUM_ISSUES','HAS_DAMAGE','HAS_CREW',
            'HAS_ADDON','HAS_PAYMENT','HAS_DELAY','HAS_RNR']
print(new_cols)

New columns added:
['IS_DETRACTOR', 'CLASS_NUM', 'FCR_FLAG', 'OTA_BREACH', 'IS_MODIFICATION', 'IS_PEAK', 'IS_NOLIFT', 'TOTAL_FLOOR', 'TAT_HRS', 'CFT_DELTA', 'VENDOR_TIER_NUM', 'BHK_SIZE', 'NUM_ISSUES', 'HAS_DAMAGE', 'HAS_CREW', 'HAS_ADDON', 'HAS_PAYMENT', 'HAS_DELAY', 'HAS_RNR']


In [36]:
numeric_features = [
    'TOTAL_TICKETS', 'CUSTOMER_TICKETS', 'VENDOR_SUPERVISOR_TICKETS',
    'SPRINKLR_SESSION_COUNT', 'TAT_HRS', 'NUM_ISSUES',
    'FINAL_FARE', 'BHK_SIZE', 'CFT_DELTA', 'DRY_RUN', 'TOTAL_FLOOR'
]
print("\n=== MEAN by CLASSIFICATION ===")
print(df.groupby('CLASSIFICATION')[numeric_features].mean().round(2).T)

print("\n=== MEDIAN by CLASSIFICATION ===")
print(df.groupby('CLASSIFICATION')[numeric_features].median().round(2).T)




=== MEAN by CLASSIFICATION ===
CLASSIFICATION             Detractor  Neutral  Promoter
TOTAL_TICKETS                   2.89     1.16      0.48
CUSTOMER_TICKETS                1.45     0.68      0.30
VENDOR_SUPERVISOR_TICKETS       0.24     0.18      0.09
SPRINKLR_SESSION_COUNT          4.54     1.93      0.98
TAT_HRS                        56.44    12.71      0.81
NUM_ISSUES                      2.51     1.11      0.48
FINAL_FARE                   6710.24  5552.96   5079.83
BHK_SIZE                        1.31     1.07      0.90
CFT_DELTA                      35.53    25.83     14.47
DRY_RUN                        12.64     6.62      7.83
TOTAL_FLOOR                     1.77     2.23      2.06

=== MEDIAN by CLASSIFICATION ===
CLASSIFICATION             Detractor  Neutral  Promoter
TOTAL_TICKETS                   2.00     1.00      0.00
CUSTOMER_TICKETS                1.00     0.00      0.00
VENDOR_SUPERVISOR_TICKETS       0.00     0.00      0.00
SPRINKLR_SESSION_COUNT          2.00  

In [37]:
binary_features = {
    'FCR Achieved':    'FCR_FLAG',
    'OTA Breach':      'OTA_BREACH',
    'Order Modified':  'IS_MODIFICATION',
    'Peak Day':        'IS_PEAK',
    'No Lift':         'IS_NOLIFT',
    'Goods Damaged':   'HAS_DAMAGE',
    'Crew Behaviour':  'HAS_CREW',
    'Add-on Issue':    'HAS_ADDON',
    'Payment Issue':   'HAS_PAYMENT',
    'RNR':             'HAS_RNR',
}

print("\n=== % with Flag=1 by CLASSIFICATION ===")
for name, col in binary_features.items():
    row = df.groupby('CLASSIFICATION')[col].mean().mul(100).round(1)
    print(f"  {name:<20} {row.to_dict()}")


=== % with Flag=1 by CLASSIFICATION ===
  FCR Achieved         {'Detractor': 48.1, 'Neutral': 37.5, 'Promoter': 20.1}
  OTA Breach           {'Detractor': 13.6, 'Neutral': 10.7, 'Promoter': 5.4}
  Order Modified       {'Detractor': 74.9, 'Neutral': 74.1, 'Promoter': 64.6}
  Peak Day             {'Detractor': 42.9, 'Neutral': 41.2, 'Promoter': 36.5}
  No Lift              {'Detractor': 49.1, 'Neutral': 51.3, 'Promoter': 48.7}
  Goods Damaged        {'Detractor': 17.7, 'Neutral': 3.7, 'Promoter': 0.9}
  Crew Behaviour       {'Detractor': 20.5, 'Neutral': 5.8, 'Promoter': 1.0}
  Add-on Issue         {'Detractor': 16.6, 'Neutral': 9.5, 'Promoter': 4.0}
  Payment Issue        {'Detractor': 8.4, 'Neutral': 2.0, 'Promoter': 0.4}
  RNR                  {'Detractor': 27.3, 'Neutral': 4.6, 'Promoter': 0.1}


In [38]:
features = [
    {'col': 'TOTAL_TICKETS',          'label': 'Total Tickets',          'clip': 10,    'bins': list(range(0,12)),                                    'bw': 0.25},
    {'col': 'CUSTOMER_TICKETS',       'label': 'Customer Tickets',       'clip': 8,     'bins': list(range(0,10)),                                    'bw': 0.25},
    {'col': 'FCR_TICKETS',            'label': 'FCR Tickets',            'clip': 6,     'bins': list(range(0,8)),                                     'bw': 0.25},
    {'col': 'SPRINKLR_SESSION_COUNT', 'label': 'Sprinklr Sessions',      'clip': 16,    'bins': list(range(0,18)),                                    'bw': 0.25},
    {'col': 'TAT_HRS',                'label': 'Resolution TAT (hrs)',   'clip': 200,   'bins': [0,6,12,24,48,72,96,120,150,200],                     'bw': 0.18},
    {'col': 'OTA_BREACH_TAT_MINUTES', 'label': 'OTA Breach TAT (mins)',  'clip': 300,   'bins': [0,30,60,90,120,180,240,300],                         'bw': 0.20},
    {'col': 'NUM_ISSUES',             'label': 'Num Distinct Issues',    'clip': 8,     'bins': list(range(0,10)),                                    'bw': 0.25},
    {'col': 'FINAL_FARE',             'label': 'Final Fare (₹)',         'clip': 25000, 'bins': [0,2000,4000,6000,8000,10000,15000,20000,25000],      'bw': 0.20},
    {'col': 'CFT_DELTA',              'label': 'CFT Delta (mins)',       'clip': 300,   'bins': [0,50,100,150,200,250,300],                           'bw': 0.20},
    {'col': 'DRY_RUN',                'label': 'Dry Run Distance (km)',  'clip': 35,    'bins': [0,5,10,15,20,25,30,35],                              'bw': 0.20},
    {'col': 'PICKUP_FLOOR',           'label': 'Pickup Floor',           'clip': 8,     'bins': list(range(0,10)),                                    'bw': 0.25},
    {'col': 'DROP_FLOOR',             'label': 'Drop Floor',             'clip': 8,     'bins': list(range(0,10)),                                    'bw': 0.25},
]

In [39]:
BG, PANEL, BORDER = '#0d1117', '#161b22', '#30363d'
WHITE, SUBTEXT    = '#e6edf3', '#8b949e'
CLASS_COLORS = {
    'Promoter':  '#3fb950',   # green
    'Neutral':   '#ffd166',   # yellow
    'Detractor': '#f85149',   # red
}
C_BARS_SAFE = '#334466'       # volume bar — safe zone
C_BARS_WARN = '#ffd166'       # volume bar — watch zone (20–40% det rate)
C_BARS_HIGH = '#f85149'       # volume bar — high risk (40%+ det rate)
C_RATE_LINE = '#f85149'       # detractor rate line

plt.rcParams.update({
    'font.family':       'DejaVu Sans',
    'text.color':        WHITE,
    'axes.facecolor':    PANEL,
    'figure.facecolor':  BG,
    'axes.edgecolor':    BORDER,
    'axes.labelcolor':   WHITE,
    'xtick.color':       SUBTEXT,
    'ytick.color':       SUBTEXT,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'grid.color':        BORDER,
    'grid.alpha':        0.3,
})


# ============================================================
# 4. PLOT
# ============================================================

n_feats = len(features)
fig, axes = plt.subplots(
    n_feats, 2,
    figsize=(26, n_feats * 4.8),
    facecolor=BG
)

fig.text(0.012, 0.999,
         'KDE  vs  DETRACTOR RATE — All Relevant Numerical Features',
         fontsize=17, fontweight='bold', color=C_RATE_LINE, va='top')
fig.text(0.012, 0.996,
         'LEFT: density shape per class (KDE)  ·  '
         'RIGHT: bars = order volume, line = % detractors  ·  n=4,744',
         fontsize=10, color=SUBTEXT, va='top')


for i, feat in enumerate(features):
    col   = feat['col']
    label = feat['label']
    clip  = feat['clip']
    bins  = feat['bins']
    bw    = feat['bw']

    ax_kde  = axes[i][0]
    ax_dual = axes[i][1]

    # OTA breach: only use orders where a breach actually happened
    source = df[df[col].notna()].copy() if col == 'OTA_BREACH_TAT_MINUTES' else df
    note   = f' (breach orders only, n={len(source)})' if col == 'OTA_BREACH_TAT_MINUTES' else ''

    # ── LEFT: KDE ──────────────────────────────────────────
    x_grid  = np.linspace(0, clip, 500)
    y_maxes = []

    for cls in ['Promoter', 'Neutral', 'Detractor']:
        data = source[source['CLASSIFICATION'] == cls][col].dropna().clip(upper=clip).values

        # Skip if too few points or no variance (KDE would crash or be meaningless)
        if len(data) < 10 or data.std() < 1e-6:
            continue

        kde   = gaussian_kde(data, bw_method=bw)
        y     = kde(x_grid)
        color = CLASS_COLORS[cls]
        med   = float(np.median(data))
        y_maxes.append(y.max())

        ax_kde.fill_between(x_grid, y, alpha=0.15, color=color)
        ax_kde.plot(x_grid, y, color=color, lw=2.2,
                    label=f'{cls}  (med={med:.1f})', zorder=4)
        ax_kde.axvline(med, color=color, lw=1.3, ls='--', alpha=0.7, zorder=3)

    y_top = max(y_maxes) * 1.20 if y_maxes else 1
    x_ticks = bins[::max(1, len(bins) // 7)]   # at most 7 ticks

    ax_kde.set_xlim(0, clip)
    ax_kde.set_ylim(0, y_top)
    ax_kde.set_xticks(x_ticks)
    ax_kde.set_xticklabels([str(v) for v in x_ticks], fontsize=9)
    ax_kde.set_xlabel(label, fontsize=10, labelpad=4)
    ax_kde.set_ylabel('Density', fontsize=9)
    ax_kde.tick_params(axis='y', left=False, labelleft=False)
    ax_kde.set_title(f'{label}{note}', fontsize=10.5, fontweight='bold',
                     color=WHITE, pad=5)
    ax_kde.legend(fontsize=8, loc='upper right',
                  framealpha=0.15, edgecolor=BORDER)
    ax_kde.grid(axis='x', alpha=0.2)

    # ── RIGHT: Dual-axis ───────────────────────────────────
    src = source.copy()
    src[col] = src[col].clip(upper=clip)

    bin_labels = [f'{bins[j]}–{bins[j+1]}' for j in range(len(bins) - 1)]
    src['_BIN'] = pd.cut(src[col], bins=bins, labels=bin_labels, include_lowest=True)

    grp = src.groupby('_BIN', observed=True).agg(
        total     = ('IS_DETRACTOR', 'count'),
        det_count = ('IS_DETRACTOR', 'sum'),
    )
    grp['det_pct'] = grp['det_count'] / grp['total'] * 100
    grp = grp[grp['total'] >= 3]   # drop bins with too few orders to be reliable

    x_pos = np.arange(len(grp))

    # Bar colour encodes risk level
    bar_colors = [
        C_BARS_HIGH if p >= 40 else (C_BARS_WARN if p >= 20 else C_BARS_SAFE)
        for p in grp['det_pct']
    ]

    ax_dual.bar(x_pos, grp['total'], color=bar_colors,
                edgecolor=BG, lw=0.8, width=0.72, alpha=0.75, zorder=2)

    # Volume label on each bar
    for xi, (tot, dp) in enumerate(zip(grp['total'], grp['det_pct'])):
        if tot > 30:
            ax_dual.text(xi, tot + grp['total'].max() * 0.01,
                         f'{tot:,}', ha='center', fontsize=7.5,
                         color=SUBTEXT, va='bottom')

    ax_dual.set_ylabel('Order volume', fontsize=9, color=SUBTEXT)
    ax_dual.set_ylim(0, grp['total'].max() * 1.35)

    # Twin y-axis — detractor rate line
    ax_rate = ax_dual.twinx()
    ax_rate.plot(x_pos, grp['det_pct'], 'o-',
                 color=C_RATE_LINE, lw=2.5, markersize=6, zorder=5)

    for xi, (dp, tot) in enumerate(zip(grp['det_pct'], grp['total'])):
        if tot >= 3:
            ax_rate.text(xi, dp + 1.5, f'{dp:.0f}%',
                         ha='center', fontsize=8,
                         color=C_RATE_LINE, fontweight='bold')

    
    ax_rate.set_ylabel('% Detractors', fontsize=9, color=C_RATE_LINE)
    ax_rate.tick_params(colors=C_RATE_LINE)
    ax_rate.set_ylim(0, min(110, grp['det_pct'].max() * 1.5 + 5))
    ax_rate.spines['right'].set_edgecolor(C_RATE_LINE)

    ax_dual.set_xticks(x_pos)
    ax_dual.set_xticklabels(grp.index.tolist(), fontsize=8,
                             rotation=30, ha='right')
    ax_dual.set_xlabel(label, fontsize=10, labelpad=4)
    ax_dual.set_title(f'{label}{note}', fontsize=10.5, fontweight='bold',
                      color=WHITE, pad=5)
    ax_dual.grid(axis='y', alpha=0.2)

    # Legend
    legend_els = [
        mpatches.Patch(facecolor=C_BARS_SAFE, alpha=0.75, label='Volume — safe zone (<20% det)'),
        mpatches.Patch(facecolor=C_BARS_WARN, alpha=0.75, label='Volume — watch zone (20–40%)'),
        mpatches.Patch(facecolor=C_BARS_HIGH, alpha=0.75, label='Volume — high risk (>40%)'),
        plt.Line2D([0], [0], color=C_RATE_LINE, lw=2,
                   marker='o', label='% Detractors (right axis)'),
    ]
    ax_dual.legend(handles=legend_els, fontsize=7.5, loc='upper right',
                   framealpha=0.15, edgecolor=BORDER)


plt.subplots_adjust(left=0.04, right=0.96, top=0.993, bottom=0.02,
                    hspace=0.70, wspace=0.30)
plt.savefig('step3_kde_vs_detrate_all.png', dpi=150,
            bbox_inches='tight', facecolor=BG)
plt.close()
print("Saved: step3_kde_vs_detrate_all.png")


Saved: step3_kde_vs_detrate_all.png


In [40]:
det       = df[df['IS_DETRACTOR'] == 1].copy()
TOTAL_DET = len(det)
C_DET     = '#f85149'
C_LINE = '#58a6ff'
C_80   = '#ffd166'


def plot_composition(ax, src, feat):
    col, label = feat['col'], feat['label']
    bins = feat['bins']
    rot  = feat.get('rot', 0)

    tmp = src.copy()
    tmp[col] = tmp[col].clip(upper=bins[-1])
    bin_labels = [f'{bins[j]}–{bins[j+1]}' for j in range(len(bins) - 1)]
    tmp['_BIN'] = pd.cut(tmp[col], bins=bins, labels=bin_labels, include_lowest=True)

    grp = tmp.groupby('_BIN', observed=True).size().reset_index(name='n')
    grp['pct']   = grp['n'] / TOTAL_DET * 100
    grp['cumul'] = grp['pct'].cumsum()

    x = np.arange(len(grp))

    for xi, (h, cumul) in enumerate(zip(grp['pct'], grp['cumul'])):
        ax.bar(xi, h, color=C_DET, alpha=1.0 if cumul <= 80 else 0.35,
               edgecolor=BG, width=0.72, zorder=3)

    for xi, (pct, n) in enumerate(zip(grp['pct'], grp['n'])):
        if pct >= 1:
            ax.text(xi, pct + 0.4, f'{pct:.1f}%', ha='center', fontsize=7, color=WHITE, fontweight='bold')
        ax.text(xi, -3.5, f'n={n}', ha='center', fontsize=6, color=SUBTEXT)

    ax.set_ylim(-5, grp['pct'].max() * 1.45)
    ax.set_ylabel('% of all detractors', fontsize=8.5, color=C_DET)
    ax.set_xticks(x)
    ax.set_xticklabels(grp['_BIN'].tolist(), fontsize=8, rotation=rot, ha='right' if rot else 'center')
    ax.grid(axis='y', alpha=0.2)

    ax2 = ax.twinx()
    ax2.plot(x, grp['cumul'], 'o-', color=C_LINE, lw=2.2, markersize=5, zorder=5)
    for xi, cv in enumerate(grp['cumul']):
        ax2.text(xi, cv + 1.5, f'{cv:.0f}%', ha='center', fontsize=7, color=C_LINE, fontweight='bold')
    ax2.axhline(80, color=C_80, lw=1.2, ls=':', alpha=0.8)
    ax2.text(len(x) - 0.4, 81, '80%', color=C_80, fontsize=7.5, ha='right')
    ax2.set_ylim(0, 118)
    ax2.set_ylabel('Cumulative %', fontsize=8.5, color=C_LINE)
    ax2.tick_params(colors=C_LINE, labelsize=8)
    ax2.spines['right'].set_edgecolor(C_LINE)

    ax.set_title(label, fontsize=11, fontweight='bold', color=WHITE, pad=6)


# ── Plot
n_cols = 3
n_rows = int(np.ceil(len(features) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(26, n_rows * 7), facecolor=BG)
axes = axes.flatten()

fig.text(0.012, 0.999,
         f'DETRACTOR COMPOSITION — Of {TOTAL_DET} Detractors, What % Had Each Value?',
         fontsize=17, fontweight='bold', color=C_DET, va='top')
fig.text(0.012, 0.996,
         'Bars = % of detractors at that bin  ·  Blue line = cumulative %  ·  Faded = tail past 80%',
         fontsize=10.5, color=SUBTEXT, va='top')

for i, feat in enumerate(features):
    src = det[det[feat['col']].notna()] if feat['col'] == 'OTA_BREACH_TAT_MINUTES' else det
    plot_composition(axes[i], src, feat)

for j in range(len(features), len(axes)):
    axes[j].set_visible(False)

ax_leg = fig.add_axes([0.55, 0.003, 0.43, 0.025])
ax_leg.axis('off')
ax_leg.legend(handles=[
    mpatches.Patch(color=C_DET,  alpha=0.9,  label='% of detractors (bars)'),
    mpatches.Patch(color=C_DET,  alpha=0.35, label='Tail — past 80%'),
    Line2D([0], [0], color=C_LINE, lw=2, marker='o', markersize=5, label='Cumulative %'),
    Line2D([0], [0], color=C_80,  lw=1.5, ls=':', label='80% threshold'),
], fontsize=9, loc='center', ncol=4, framealpha=0.0)

plt.subplots_adjust(left=0.05, right=0.96, top=0.992, bottom=0.04, hspace=0.70, wspace=0.35)
plt.savefig('step5_detractor_composition.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.close()
print("Saved: step5_detractor_composition.png")


Saved: step5_detractor_composition.png


In [41]:
pcts = [10, 25, 50, 75, 90, 95, 99]

# ── Numeric features from config
cols = [(f['label'], f['col']) for f in features]

print(f"{'Feature':<28} {'Class':<12}", end='')
for p in pcts:
    print(f"  P{p:>2}", end='')
print()
print('-' * 95)

for label, col in cols:
    for cls in ['Promoter', 'Neutral', 'Detractor']:
        data = df[df['CLASSIFICATION'] == cls][col].dropna()
        vals = np.percentile(data, pcts)
        print(f"{label:<28} {cls:<12}", end='')
        for v in vals:
            print(f"  {v:>5.1f}", end='')
        print()
    print()

Feature                      Class         P10  P25  P50  P75  P90  P95  P99
-----------------------------------------------------------------------------------------------
Total Tickets                Promoter        0.0    0.0    0.0    1.0    2.0    2.0    5.0
Total Tickets                Neutral         0.0    0.0    1.0    2.0    3.0    4.0    6.0
Total Tickets                Detractor       1.0    1.0    2.0    4.0    6.0    7.0   10.0

Customer Tickets             Promoter        0.0    0.0    0.0    0.0    1.0    2.0    4.0
Customer Tickets             Neutral         0.0    0.0    0.0    1.0    2.0    3.0    4.0
Customer Tickets             Detractor       0.0    0.0    1.0    2.0    4.0    5.0    6.9

FCR Tickets                  Promoter        0.0    0.0    0.0    0.0    1.0    1.0    3.0
FCR Tickets                  Neutral         0.0    0.0    0.0    1.0    2.0    2.0    4.0
FCR Tickets                  Detractor       0.0    0.0    0.0    1.0    2.0    3.0    5.0

Sprin

In [42]:
print("\nDETRACTOR PERCENTILES:")
print(f"{'Feature':<28}", end='')
for p in pcts:
    print(f"  P{p:>2}", end='')
print()
print('-' * 77)

for label, col in cols:
    data = det[col].dropna()
    vals = np.percentile(data, pcts)
    print(f"{label:<28}", end='')
    for v in vals:
        print(f"  {v:>5.1f}", end='')
    print()



DETRACTOR PERCENTILES:
Feature                       P10  P25  P50  P75  P90  P95  P99
-----------------------------------------------------------------------------
Total Tickets                   1.0    1.0    2.0    4.0    6.0    7.0   10.0
Customer Tickets                0.0    0.0    1.0    2.0    4.0    5.0    6.9
FCR Tickets                     0.0    0.0    0.0    1.0    2.0    3.0    5.0
Sprinklr Sessions               0.0    0.0    2.0    6.0   12.0   16.0   27.6
Resolution TAT (hrs)            0.9    3.4   11.8   73.3  165.2  278.1  431.9
OTA Breach TAT (mins)           2.9   13.0   24.5   76.8  158.1  238.5  677.6
Num Distinct Issues             0.0    1.0    2.0    4.0    5.0    7.0    9.0
Final Fare (₹)                2150.5  3411.0  5824.9  8396.1  11849.8  15077.9  26331.4
CFT Delta (mins)                0.0    0.0    0.0   48.0  140.0  204.4  360.5
Dry Run Distance (km)           0.0    0.0    1.8   12.4   19.9   24.6   41.1
Pickup Floor                    0.0    0.0  

## 4. Lift × Order Share — Impact Ranking

In [43]:
overall_det_rate = df['IS_DETRACTOR'].mean()
total_orders     = len(df)

impact_rows = []
for feat in features:
    col, label, bins, clip = feat['col'], feat['label'], feat['bins'], feat['clip']
    src = df[df[col].notna()].copy() if col == 'OTA_BREACH_TAT_MINUTES' else df.copy()
    src[col] = src[col].clip(upper=clip)
    bin_labels = [f'{bins[j]}\u2013{bins[j+1]}' for j in range(len(bins) - 1)]
    src['_BIN'] = pd.cut(src[col], bins=bins, labels=bin_labels, include_lowest=True)

    grp = src.groupby('_BIN', observed=True).agg(
        orders    = ('IS_DETRACTOR', 'count'),
        det_count = ('IS_DETRACTOR', 'sum'),
    )
    grp = grp[grp['orders'] >= 3]
    grp['det_rate']     = grp['det_count'] / grp['orders']
    grp['order_share']  = grp['orders'] / total_orders
    grp['lift']         = grp['det_rate'] / overall_det_rate
    grp['lift_x_share'] = grp['lift'] * grp['order_share']

    for bin_name, row in grp.iterrows():
        impact_rows.append({
            'Feature':       label,
            'Bin':           str(bin_name),
            'Orders':        int(row['orders']),
            'Det Rate %':    round(row['det_rate'] * 100, 1),
            'Order Share %': round(row['order_share'] * 100, 1),
            'Lift':          round(row['lift'], 2),
            'Lift\u00d7Share': round(row['lift_x_share'], 4),
        })

impact_df = (pd.DataFrame(impact_rows)
               .sort_values('Lift\u00d7Share', ascending=False)
               .reset_index(drop=True))

print(f"Baseline det rate: {overall_det_rate*100:.1f}%  |  Total orders: {total_orders:,}\n")
print("=== TOP 30 HIGH-IMPACT BINS  (Lift \u00d7 Order Share, descending) ===")
print(impact_df.head(30).to_string(index=False))


Baseline det rate: 10.8%  |  Total orders: 4,744

=== TOP 30 HIGH-IMPACT BINS  (Lift × Order Share, descending) ===
              Feature         Bin  Orders  Det Rate %  Order Share %  Lift  Lift×Share
           Drop Floor         0–1    3541        11.7           74.6  1.09      0.8109
          FCR Tickets         0–1    4409         9.1           92.9  0.84      0.7817
         Pickup Floor         0–1    3454        10.7           72.8  0.99      0.7193
     CFT Delta (mins)        0–50    3605         9.5           76.0  0.88      0.6667
     Customer Tickets         0–1    4258         7.5           89.8  0.70      0.6257
Dry Run Distance (km)         0–5    2600        11.2           54.8  1.03      0.5653
    Sprinklr Sessions         0–1    3623         6.1           76.4  0.56      0.4308
  Num Distinct Issues         0–1    3936         5.4           83.0  0.50      0.4133
 Resolution TAT (hrs)         0–6    4259         4.3           89.8  0.40      0.3548
        Total 

In [44]:
top_n = 25
top   = impact_df.head(top_n).copy().reset_index(drop=True)
top['label'] = top['Feature'] + '  |  ' + top['Bin']

fig, ax = plt.subplots(figsize=(16, top_n * 0.52 + 2), facecolor=BG)
ax.set_facecolor(PANEL)

bar_colors = [
    C_BARS_HIGH if l >= 2.0 else (C_BARS_WARN if l >= 1.2 else C_BARS_SAFE)
    for l in top['Lift']
]
ax.barh(range(top_n), top['Lift\u00d7Share'] * 100,
        color=bar_colors, alpha=0.88, edgecolor=BG, height=0.68, zorder=3)

x_max = top['Lift\u00d7Share'].max() * 100
for yi, row in top.iterrows():
    ax.text(
        row['Lift\u00d7Share'] * 100 + x_max * 0.015, yi,
        f"lift={row['Lift']:.1f}x   det={row['Det Rate %']:.0f}%   share={row['Order Share %']:.1f}%",
        va='center', fontsize=8.5, color=SUBTEXT,
    )

ax.set_yticks(range(top_n))
ax.set_yticklabels(top['label'], fontsize=9.5)
ax.set_xlabel('Lift \u00d7 Order Share  (\u00d7100)', fontsize=11, color=WHITE, labelpad=6)
ax.set_xlim(0, x_max * 2.3)
ax.invert_yaxis()
ax.set_title(
    f'Top {top_n} High-Impact Bins \u2014 Lift \u00d7 Order Share\n'
    f'Baseline det rate: {overall_det_rate*100:.1f}%  \u00b7  '
    f'Lift = bin det rate \u00f7 baseline  \u00b7  bars coloured by lift tier',
    fontsize=13, fontweight='bold', color=WHITE, pad=10,
)
ax.grid(axis='x', alpha=0.25)

legend_els = [
    mpatches.Patch(facecolor=C_BARS_HIGH, alpha=0.88, label='Lift \u2265 2.0\u00d7  (high risk)'),
    mpatches.Patch(facecolor=C_BARS_WARN, alpha=0.88, label='Lift 1.2\u20132.0\u00d7  (watch)'),
    mpatches.Patch(facecolor=C_BARS_SAFE, alpha=0.88, label='Lift < 1.2\u00d7  (near baseline)'),
]
ax.legend(handles=legend_els, fontsize=9, loc='lower right',
          framealpha=0.15, edgecolor=BORDER)

plt.tight_layout()
plt.savefig('step7_lift_x_share_priority.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.close()
print("Saved: step7_lift_x_share_priority.png")


Saved: step7_lift_x_share_priority.png


## 5. KDE vs Detractor Rate (Lift-coloured, Lift×Share annotated)

In [45]:
n_feats = len(features)
fig, axes = plt.subplots(n_feats, 2, figsize=(26, n_feats * 4.8), facecolor=BG)

fig.text(0.012, 0.999,
         'KDE vs DETRACTOR RATE \u2014 Bars coloured by Lift, annotated with Lift\u00d7Share score',
         fontsize=17, fontweight='bold', color=C_RATE_LINE, va='top')
fig.text(0.012, 0.996,
         f'LEFT: KDE per class  \u00b7  RIGHT: bars = lift tier, '
         f'red line = det rate, dotted = baseline ({overall_det_rate*100:.1f}%), '
         f'bar label = Lift\u00d7Share score  \u00b7  n={total_orders:,}',
         fontsize=10, color=SUBTEXT, va='top')

for i, feat in enumerate(features):
    col, label, clip = feat['col'], feat['label'], feat['clip']
    bins, bw         = feat['bins'], feat['bw']
    ax_kde  = axes[i][0]
    ax_dual = axes[i][1]

    source = df[df[col].notna()].copy() if col == 'OTA_BREACH_TAT_MINUTES' else df
    note   = f' (breach orders only, n={len(source)})' if col == 'OTA_BREACH_TAT_MINUTES' else ''

    # ── LEFT: KDE (unchanged) ──────────────────────────────────────────
    x_grid, y_maxes = np.linspace(0, clip, 500), []
    for cls in ['Promoter', 'Neutral', 'Detractor']:
        data = source[source['CLASSIFICATION'] == cls][col].dropna().clip(upper=clip).values
        if len(data) < 10 or data.std() < 1e-6:
            continue
        kde   = gaussian_kde(data, bw_method=bw)
        y     = kde(x_grid)
        color = CLASS_COLORS[cls]
        med   = float(np.median(data))
        y_maxes.append(y.max())
        ax_kde.fill_between(x_grid, y, alpha=0.15, color=color)
        ax_kde.plot(x_grid, y, color=color, lw=2.2,
                    label=f'{cls}  (med={med:.1f})', zorder=4)
        ax_kde.axvline(med, color=color, lw=1.3, ls='--', alpha=0.7, zorder=3)

    y_top   = max(y_maxes) * 1.20 if y_maxes else 1
    x_ticks = bins[::max(1, len(bins) // 7)]
    ax_kde.set_xlim(0, clip);  ax_kde.set_ylim(0, y_top)
    ax_kde.set_xticks(x_ticks)
    ax_kde.set_xticklabels([str(v) for v in x_ticks], fontsize=9)
    ax_kde.set_xlabel(label, fontsize=10, labelpad=4)
    ax_kde.set_ylabel('Density', fontsize=9)
    ax_kde.tick_params(axis='y', left=False, labelleft=False)
    ax_kde.set_title(f'{label}{note}', fontsize=10.5, fontweight='bold', color=WHITE, pad=5)
    ax_kde.legend(fontsize=8, loc='upper right', framealpha=0.15, edgecolor=BORDER)
    ax_kde.grid(axis='x', alpha=0.2)

    # ── RIGHT: Dual-axis with lift coloring & Lift\u00d7Share annotation ──
    src = source.copy()
    src[col] = src[col].clip(upper=clip)
    bin_labels = [f'{bins[j]}\u2013{bins[j+1]}' for j in range(len(bins) - 1)]
    src['_BIN'] = pd.cut(src[col], bins=bins, labels=bin_labels, include_lowest=True)

    grp = src.groupby('_BIN', observed=True).agg(
        total     = ('IS_DETRACTOR', 'count'),
        det_count = ('IS_DETRACTOR', 'sum'),
    )
    grp['det_pct']      = grp['det_count'] / grp['total'] * 100
    grp['order_share']  = grp['total'] / total_orders
    grp['lift']         = (grp['det_pct'] / 100) / overall_det_rate
    grp['lift_x_share'] = grp['lift'] * grp['order_share']
    grp = grp[grp['total'] >= 3]

    x_pos = np.arange(len(grp))

    # Colour by lift tier
    bar_colors = [
        C_BARS_HIGH if l >= 2.0 else (C_BARS_WARN if l >= 1.2 else C_BARS_SAFE)
        for l in grp['lift']
    ]
    ax_dual.bar(x_pos, grp['total'], color=bar_colors,
                edgecolor=BG, lw=0.8, width=0.72, alpha=0.75, zorder=2)

    vol_max = grp['total'].max()
    for xi, (tot, lxs) in enumerate(zip(grp['total'], grp['lift_x_share'])):
        # volume label on tall bars
        if tot > 30:
            ax_dual.text(xi, tot + vol_max * 0.01, f'{tot:,}',
                         ha='center', fontsize=7, color=SUBTEXT, va='bottom')
        # Lift\u00d7Share score inside bar (rotated)
        ax_dual.text(xi, vol_max * 0.03, f'L\u00d7S={lxs*100:.1f}',
                     ha='center', fontsize=6.5, color=WHITE, alpha=0.9,
                     va='bottom', rotation=90)

    ax_dual.set_ylabel('Order volume', fontsize=9, color=SUBTEXT)
    ax_dual.set_ylim(0, vol_max * 1.50)

    # Det rate line + baseline reference
    ax_rate = ax_dual.twinx()
    ax_rate.plot(x_pos, grp['det_pct'], 'o-',
                 color=C_RATE_LINE, lw=2.5, markersize=6, zorder=5)
    ax_rate.axhline(overall_det_rate * 100, color=C_RATE_LINE,
                    lw=1.0, ls=':', alpha=0.55, zorder=4)
    for xi, (dp, tot) in enumerate(zip(grp['det_pct'], grp['total'])):
        if tot >= 3:
            ax_rate.text(xi, dp + 1.5, f'{dp:.0f}%',
                         ha='center', fontsize=8,
                         color=C_RATE_LINE, fontweight='bold')

    ax_rate.set_ylabel('% Detractors', fontsize=9, color=C_RATE_LINE)
    ax_rate.tick_params(colors=C_RATE_LINE)
    ax_rate.set_ylim(0, min(110, grp['det_pct'].max() * 1.5 + 5))
    ax_rate.spines['right'].set_edgecolor(C_RATE_LINE)

    ax_dual.set_xticks(x_pos)
    ax_dual.set_xticklabels(grp.index.tolist(), fontsize=8, rotation=30, ha='right')
    ax_dual.set_xlabel(label, fontsize=10, labelpad=4)
    ax_dual.set_title(f'{label}{note}', fontsize=10.5, fontweight='bold', color=WHITE, pad=5)
    ax_dual.grid(axis='y', alpha=0.2)

    legend_els = [
        mpatches.Patch(facecolor=C_BARS_SAFE, alpha=0.75, label='Lift < 1.2\u00d7  (near baseline)'),
        mpatches.Patch(facecolor=C_BARS_WARN, alpha=0.75, label='Lift 1.2\u20132.0\u00d7  (watch)'),
        mpatches.Patch(facecolor=C_BARS_HIGH, alpha=0.75, label='Lift \u2265 2.0\u00d7  (high risk)'),
        plt.Line2D([0], [0], color=C_RATE_LINE, lw=2, marker='o', label='% Detractors'),
        plt.Line2D([0], [0], color=C_RATE_LINE, lw=1, ls=':', label=f'Baseline ({overall_det_rate*100:.1f}%)'),
    ]
    ax_dual.legend(handles=legend_els, fontsize=7.5, loc='upper right',
                   framealpha=0.15, edgecolor=BORDER)

plt.subplots_adjust(left=0.04, right=0.96, top=0.993, bottom=0.02,
                    hspace=0.70, wspace=0.30)
plt.savefig('step8_kde_lift_x_share.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.close()
print("Saved: step8_kde_lift_x_share.png")


Saved: step8_kde_lift_x_share.png


Bivariate analysis

In [46]:
binary_cols = {
    'FCR Flag':       'FCR_FLAG',
    'OTA Breach':     'OTA_BREACH',
    'Order Modified': 'IS_MODIFICATION',
    'Peak Day':       'IS_PEAK',
    'No Lift':        'IS_NOLIFT',
    'Goods Damaged':  'HAS_DAMAGE',
    'Crew Behaviour': 'HAS_CREW',
    'Add-on Issue':   'HAS_ADDON',
    'Payment Issue':  'HAS_PAYMENT',
    'RNR':            'HAS_RNR',
}

all_features = [(f['label'], f['col']) for f in features] + list(binary_cols.items())

results = []
for label, col in all_features:
    if col not in df.columns:
        continue
    r, p = spearmanr(df[col].fillna(0), df['IS_DETRACTOR'])
    results.append({
        'feature': label,
        'r':       round(r, 4),
        'p':       round(p, 6),
        'sig':     '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns')),
    })

corr_df = pd.DataFrame(results).sort_values('r', ascending=False)
print(corr_df.to_string(index=False))

# ── Plot
fig, ax = plt.subplots(figsize=(10, len(corr_df) * 0.40 + 1), facecolor=BG)

corr_sorted = corr_df.sort_values('r', ascending=True)
colors = [C_DET if r > 0 else '#58a6ff' for r in corr_sorted['r']]

ax.barh(range(len(corr_sorted)), corr_sorted['r'],
        color=colors, edgecolor=BG, height=0.7, alpha=0.88)

ax.axvline(0,   color=WHITE, lw=1.2, alpha=0.5)
ax.axvline(0.3, color=C_DET, lw=0.8, ls=':')

for yi, (_, row) in enumerate(corr_sorted.iterrows()):
    xoff = 0.008 if row['r'] >= 0 else -0.008
    ha   = 'left' if row['r'] >= 0 else 'right'
    ax.text(row['r'] + xoff, yi, f"{row['r']:+.3f} {row['sig']}",
            va='center', fontsize=8, color=colors[yi], fontweight='bold', ha=ha)

ax.set_yticks(range(len(corr_sorted)))
ax.set_yticklabels(corr_sorted['feature'], fontsize=9)
ax.set_xlabel('Spearman r with IS_DETRACTOR', fontsize=10)
ax.set_xlim(-0.25, 0.95)
ax.set_title('Feature Correlation with IS_DETRACTOR (full dataset, n=4,744)',
             fontsize=13, fontweight='bold', color=WHITE, pad=8)
ax.grid(axis='x', alpha=0.25)

plt.tight_layout()
plt.savefig('step6_feature_vs_detractor.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.close()
print("Saved: step6_feature_vs_detractor.png")


              feature       r        p sig
 Resolution TAT (hrs)  0.6463 0.000000 ***
        Total Tickets  0.4878 0.000000 ***
                  RNR  0.4650 0.000000 ***
  Num Distinct Issues  0.4117 0.000000 ***
       Crew Behaviour  0.3242 0.000000 ***
     Customer Tickets  0.3078 0.000000 ***
        Goods Damaged  0.3073 0.000000 ***
    Sprinklr Sessions  0.2848 0.000000 ***
        Payment Issue  0.2123 0.000000 ***
          FCR Tickets  0.2069 0.000000 ***
             FCR Flag  0.1924 0.000000 ***
         Add-on Issue  0.1617 0.000000 ***
       Final Fare (₹)  0.1309 0.000000 ***
OTA Breach TAT (mins)  0.1027 0.000000 ***
           OTA Breach  0.0980 0.000000 ***
     CFT Delta (mins)  0.0745 0.000000 ***
       Order Modified  0.0620 0.000019 ***
             Peak Day  0.0383 0.008395  **
              No Lift  0.0011 0.940473  ns
         Pickup Floor -0.0070 0.629473  ns
Dry Run Distance (km) -0.0130 0.370086  ns
           Drop Floor -0.0417 0.004043  **
Saved: step